In [1]:
import polars as pl 
import polars.selectors as cs
from datetime import datetime
import polars_ds as pds 
import requests 
import json 
import duckdb
from pathlib import Path
from src.utils import (
    scrape_eliteserien_goal_scorers_for_seasons,
    scrape_eliteserien_results_for_seasons,
)


In [2]:
seasons = [
    (2020,2021),
    (2021,2022),
    (2022,2023),
    (2023,2024),
    (2024,2025),
    (2025,2026)
]

In [10]:
def save_eliteserien_results_for_season(
    season: tuple[int, int | None],
    db_path: Path,
    delay_seconds: float = 0.5,
) -> None:
    """Scrape one Eliteserien season and store it in its own raw_eliteserien_results_<year> table."""
    year = season[1] or season[0]
    table_name = f"raw_eliteserien_results_{year}"

    data = scrape_eliteserien_results_for_seasons([season], delay_seconds=delay_seconds)
    if not data:
        print(f"No records returned for season {season}; skipping.")
        return

    df = pl.DataFrame(data).with_columns(pl.lit(datetime.now()).alias("ingested_at"))

    with duckdb.connect(str(db_path)) as connection:
        connection.register("new_data", df)
        connection.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM new_data")

    print(f"✓ Saved {len(data)} matches to {table_name}")


# Ingest results season by season so one failing season doesn't block the rest
for season in seasons:
    try:
        save_eliteserien_results_for_season(season, db_path, delay_seconds=0.5)
    except Exception as error:
        print(f"Skipping season {season}: {error}")


✓ Saved 240 matches to raw_eliteserien_results_2021
✓ Saved 240 matches to raw_eliteserien_results_2022
✓ Saved 240 matches to raw_eliteserien_results_2023
✓ Saved 240 matches to raw_eliteserien_results_2024
✓ Saved 240 matches to raw_eliteserien_results_2025
✓ Saved 168 matches to raw_eliteserien_results_2026


In [5]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print("Available tables:")
for table in table_names:
    print(f"  - {table[0]}")
con.close()

Available tables:
  - dim_teams
  - fct_league_standings
  - fct_matches
  - fct_match_statistics
  - raw_eliteserien_results_2021
  - raw_eliteserien_results_2022
  - raw_eliteserien_results_2023
  - raw_eliteserien_results_2024
  - raw_eliteserien_results_2025
  - raw_eliteserien_results_2026
  - raw_match_statistics


In [4]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("DROP TABLE IF EXISTS fct_goal_scorers").fetchall()
con.close()

In [ ]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("select * from raw_eliteserien_results_2026").arrow()
pl.from_arrow(table_names)

date,matchday,home_team,away_team,result,report_url,snapshot_at,season,ingested_at
date,i64,str,str,str,str,"datetime[μs, Europe/Oslo]",i64,datetime[μs]
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
…,…,…,…,…,…,…,…,…
2026-09-20,22,"""Vålerenga""","""Fredrikstad FK""","""1:1""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-09-20,22,"""Sandefjord""","""IK Start""","""3:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-09-20,22,"""Tromsø IL""","""HamKam""","""3:2""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546


: 